# XProf Kernel: Fine-Grained Runtime Performance Counters

This notebook is a customer-facing instruction focusing on (1) interface examples on enabling runtime performance counters, and (2) demonstration of how fine-grained runtime counters can expose the performance characteristics of optimized kernels more accurately than static analysis alone.

**Note:** To execute this demo, please connect to an Ironwood runtime.

```
Copyright 2026 Google LLC.

SPDX-License-Identifier: Apache-2.0
```

Author: Jiya Zhang, Prasanna Rengasamy

In [ ]:
import os

# Enable LLO region insertion and register LLO debug info.
os.environ["LIBTPU_INIT_ARGS"] = (
    "--xla_enable_custom_call_region_trace=true "
    "--xla_xprof_register_llo_debug_info=true"
)

print("LIBTPU_INIT_ARGS set to:", os.environ["LIBTPU_INIT_ARGS"])

## Optimized Matmul Pallas Kernel

This implementation uses a Pallas kernel to perform tiled matrix multiplication on a TPU, splitting the computation into 8 blocks along the M dimension. Key optimizations include:

*   **`pltpu.emit_pipeline`**: Used to manage data movement and execution flow efficiently.
*   Multi-Buffering: Employs triple buffering for the input matrix ($X$) and double buffering for the output matrix ($Z$) to overlap data transfers between High-Bandwidth Memory (HBM) and Vector Memory (VMEM) with execution
*   Pipelining: Ensures high hardware utilization by pre-fetching tiles for subsequent computation steps while the current tile is being processed.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import functools
from jax import random

from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

def matmul_pipeline_kernel(x_ref, y_ref, z_ref, *, bm, bk, bn):
  m, k = x_ref.shape
  _, n = y_ref.shape

  # Define the shapes of the local VMEM tiles
  x_tile_shape = (bm, bk)
  y_tile_shape = (bk, bn)
  z_tile_shape = (bm, bn)

  # Explicitly define the pipeline stages
  def body_func(x_tile_ref, y_tile_ref, z_tile_ref):
    # Retrieve grid index for the reduction dimension
    k_idx = pl.program_id(2)

    @pl.when(k_idx == 0)
    def _():
      # Overwrite VMEM with the first partial product.
      # This effectively initializes the tile without
      # a separate 'zero-fill' phase.
      with jax.named_scope("init"):
        z_tile_ref[...] = x_tile_ref[...] @ y_tile_ref[...]

    @pl.when(k_idx > 0) # => condition logic handled by ALU
    def _():
      # Accumulate subsequent partial products into
      # the existing VMEM tile.
      with jax.named_scope("acc"):
        z_tile_ref[...] += x_tile_ref[...] @ y_tile_ref[...]
          # @ => MXU
          # += => VPU (the element-wise addition of the new matrix product to the existing partial sum stored in VMEM)
          # Element-wise operations on arrays, such as custom additions, activations (ReLU, GeLU), and reductions.

  # Call emit_pipeline
  pltpu.emit_pipeline(
      body_func,
      grid=(m // bm, n // bn, k // bk),
      in_specs=[
          pl.BlockSpec(
              x_tile_shape, lambda i, j, k: (i, k),
              pipeline_mode=pl.Buffered(buffer_count=3)
              #  (Triple Buffering): allocate 3 buffers in VMEM for tiles of X.
              # This allows the TPU to aggressively overlap:
              # while buffer 1 is being used for compute, buffer 2 can be
              # receiving data from HBM, and buffer 3 can be preparing for the next load.
          ),
          pl.BlockSpec(
              y_tile_shape, lambda i, j, k: (k, j),
              pipeline_mode=pl.Buffered(buffer_count=1)
          ),
      ],
      out_specs=pl.BlockSpec(
          z_tile_shape, lambda i, j, k: (i, j),
          pipeline_mode=pl.Buffered(buffer_count=2)
      ),
      trace_scopes=True
  )(x_ref, y_ref, z_ref)

@functools.partial(jax.jit, static_argnames=('bm', 'bk', 'bn'))
def matmul_optimized(x, y, bm=1024, bk=1024, bn=4096):
  m, k = x.shape
  _, n = y.shape

  return pl.pallas_call(
      functools.partial(matmul_pipeline_kernel, bm=bm, bk=bk, bn=bn),
      out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),

      in_specs = [
          pl.BlockSpec(memory_space=pltpu.HBM),
          pl.BlockSpec(memory_space=pltpu.HBM)
      ],
      out_specs = pl.BlockSpec(memory_space=pltpu.HBM),
      compiler_params=pltpu.CompilerParams(
          vmem_limit_bytes=64 * 1024 * 1024),
      debug = False
  )(x, y)

### Enable Periodic Counter Sampling

The following cell defines profiling options with periodic counter sampling (`interval_us`). Please adjust triggering type, sample frequency, and indices as needed for your specific requirements.

In [ ]:
options = jax.profiler.ProfileOptions()
options.advanced_configuration = {
    "tpu_enable_periodic_counter_sampling": True,
    "tpu_tc_perf_counter_sampling_options": (
        'interval_us:1 scaling:0 counter_size_bits:1 '
        'indices:1 indices:3 indices:4 indices:10 indices:11 '
        'indices:31 indices:32 indices:33 indices:34 indices:35 '
        'indices:37 indices:38 indices:56 indices:57 indices:58 '
        'indices:73 indices:74 indices:75 indices:105'
    ),
    "num_tensor_cores_to_trace_per_device": 1,
}

logdir = "/tmp/tensorboard"
print(f"Profiling kernel. Logs will be saved to {logdir}...")

m, k, n = 8192, 1024, 4096
k1, k2 = random.split(random.key(0), 2)
x = random.normal(k1, (m, k), dtype=jnp.float32)
y = random.normal(k2, (k, n), dtype=jnp.float32)

# Run with Profiler
with jax.profiler.trace(logdir, profiler_options=options):
  result = matmul_optimized(x, y, bm=1024, bk=1024, bn=4096)
  jax.block_until_ready(result)

print("Profiling done. Verifying result accuracy...")
ref = jnp.dot(x, y)
np.testing.assert_allclose(result, ref, atol=5e-3)
print("Result verified!")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /tmp/tensorboard

# XProf Session Analysis

While LLO (Low-Level Optimizer) static analysis confirms the 8 iterations of vector stores defined by the Pallas kernel, runtime performance counters reveal the "actual" hardware behavior. **These runtime metrics expose details beyond what static dumps can provide, such as the effectiveness of buffering configurations and the impact of memory stalls**.

### Optimization Comparison

In the optimized version, triple buffering hides memory latency by overlapping data transfers with MXU computation. To demonstrate the value of this optimization, we performed a "reverse optimization" by setting `pipeline_mode=pl.Buffered(buffer_count=1)` for the $X$ input.

| Optimized (3-Buffered $X$) | Memory Bound (1-Buffered $X$) |
| :--- | :--- |
| ![Optimized Screenshot](https://services.google.com/fh/files/blogs/optimized_kernel.png) | ![Memory Bound Screenshot](https://services.google.com/fh/files/blogs/unoptimized_kernel.png) |
| **Runtime: 88µs** | **Runtime: 125.5µs** |

*   **Memory Bound Profile:** The 1-buffered configuration shows a significantly higher runtime and increased activity in the `count_sync_wait` counter, indicating that the MXU is frequently stalled while waiting for data from HBM.
*   **Buffering Efficiency:** The reduction in runtime from 125.5µs to 88µs validates that the multi-buffering strategy successfully pipelines HBM-to-VMEM transfers, keeping the TPU compute units utilized.